In [ ]:
# individual_model_train.py
"""
Enhanced Individual-Level Model Training for Multi-Disease Prediction
Handles 100+ features and 400K+ samples with advanced preprocessing and evaluation
"""

import pandas as pd
import numpy as np
import os
import logging
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

# ML imports
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score, 
                            recall_score, f1_score, confusion_matrix, classification_report,
                            average_precision_score, precision_recall_curve, roc_curve)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import StackingClassifier, VotingClassifier

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class IndividualModelTrainerEnhanced:
    """Enhanced trainer for individual-level data with 100+ features."""
    
    def __init__(self, modeling_data_path: str = 'data/processed/individual_full/modeling_data.pkl'):
        self.modeling_data_path = modeling_data_path
        self.modeling_data = None
        self.models = {}
        self.results = {}
        self.best_models = {}
        self.feature_importance = {}
        
    def load_data(self):
        """Load prepared modeling data."""
        logger.info(f"Loading modeling data from {self.modeling_data_path}")
        
        if not os.path.exists(self.modeling_data_path):
            logger.error(f"Modeling data not found: {self.modeling_data_path}")
            return False
        
        self.modeling_data = joblib.load(self.modeling_data_path)
        
        logger.info("=" * 70)
        logger.info("DATASET OVERVIEW")
        logger.info("=" * 70)
        logger.info(f"X_train shape: {self.modeling_data['X_train'].shape}")
        logger.info(f"X_test shape: {self.modeling_data['X_test'].shape}")
        logger.info(f"Features: {len(self.modeling_data['feature_names'])}")
        logger.info(f"Targets: {', '.join(self.modeling_data['target_names'])}")
        
        # Check class balance
        if len(self.modeling_data['target_names']) > 0:
            for i, target in enumerate(self.modeling_data['target_names']):
                y_train = self.modeling_data['y_train'][:, i]
                prevalence = y_train.mean() * 100
                logger.info(f"  {target}: {prevalence:.2f}% prevalence")
        
        return True
    
    def create_models(self, use_advanced: bool = True):
        """Create classification models optimized for individual-level data."""
        logger.info("Creating classification models...")
        
        # Base models for stacking
        base_models = [
            ('rf', RandomForestClassifier(n_estimators=100, max_depth=10, 
                                         random_state=42, n_jobs=-1)),
            ('xgb', XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3,
                                 random_state=42, n_jobs=-1, verbosity=0)),
            ('lgbm', LGBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=5,
                                   random_state=42, n_jobs=-1, verbose=-1))
        ]
        
        if use_advanced:
            self.models = {
                # Baseline models
                'Logistic_Regression': LogisticRegression(
                    max_iter=1000, random_state=42, n_jobs=-1, class_weight='balanced'
                ),
                
                # Tree-based models
                'Decision_Tree': DecisionTreeClassifier(
                    max_depth=10, min_samples_split=20, random_state=42
                ),
                'Random_Forest': RandomForestClassifier(
                    n_estimators=200, max_depth=15, min_samples_split=10,
                    random_state=42, n_jobs=-1, class_weight='balanced'
                ),
                
                # Gradient Boosting models
                'Gradient_Boosting': GradientBoostingClassifier(
                    n_estimators=150, learning_rate=0.05, max_depth=5,
                    subsample=0.8, random_state=42
                ),
                'XGBoost': XGBClassifier(
                    n_estimators=200, learning_rate=0.05, max_depth=6,
                    subsample=0.8, colsample_bytree=0.8,
                    random_state=42, n_jobs=-1, verbosity=0,
                    eval_metric='logloss', use_label_encoder=False
                ),
                'LightGBM': LGBMClassifier(
                    n_estimators=200, learning_rate=0.05, max_depth=7,
                    subsample=0.8, colsample_bytree=0.8,
                    random_state=42, n_jobs=-1, verbose=-1,
                    class_weight='balanced'
                ),
                'CatBoost': CatBoostClassifier(
                    iterations=200, learning_rate=0.05, depth=6,
                    random_state=42, verbose=False
                ),
                
                # Ensemble models
                'AdaBoost': AdaBoostClassifier(
                    n_estimators=100, learning_rate=0.5, random_state=42
                ),
                
                # Advanced models
                'SVM': SVC(
                    probability=True, random_state=42, kernel='rbf',
                    C=1.0, gamma='scale', class_weight='balanced'
                ),
                'KNN': KNeighborsClassifier(
                    n_neighbors=5, weights='distance', n_jobs=-1
                ),
                'Naive_Bayes': GaussianNB(),
                
                # Ensemble methods
                'Voting_Classifier': VotingClassifier(
                    estimators=[
                        ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
                        ('xgb', XGBClassifier(n_estimators=100, random_state=42, verbosity=0)),
                        ('lgbm', LGBMClassifier(n_estimators=100, random_state=42, verbose=-1))
                    ],
                    voting='soft',
                    n_jobs=-1
                ),
                'Stacking_Classifier': StackingClassifier(
                    estimators=base_models,
                    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
                    cv=3,
                    n_jobs=-1
                )
            }
        else:
            # Simplified model set for testing
            self.models = {
                'Logistic_Regression': LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
                'Random_Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
                'XGBoost': XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1, verbosity=0),
                'LightGBM': LGBMClassifier(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1)
            }
        
        logger.info(f"Created {len(self.models)} models")
        return self.models
    
    def train_models(self, cv_folds: int = 3, use_cv: bool = True):
        """Train all models with optional cross-validation."""
        logger.info("=" * 70)
        logger.info("TRAINING MODELS ON INDIVIDUAL-LEVEL DATA")
        logger.info("=" * 70)
        
        X_train = self.modeling_data['X_train']
        y_train = self.modeling_data['y_train']
        target_names = self.modeling_data['target_names']
        
        self.results = {
            'model_comparison': {},
            'target_results': {target: {} for target in target_names},
            'cv_scores': {},
            'training_info': {
                'n_samples': X_train.shape[0],
                'n_features': X_train.shape[1],
                'n_targets': len(target_names),
                'cv_folds': cv_folds if use_cv else 0
            }
        }
        
        for model_name, model in self.models.items():
            logger.info(f"\nTraining {model_name}...")
            
            # Train separate model for each target
            target_models = {}
            target_metrics = {}
            cv_results_all = {}
            
            for target_idx, target_name in enumerate(target_names):
                logger.info(f"  Target: {target_name}")
                
                y_train_target = y_train[:, target_idx]
                
                # Handle class imbalance
                if hasattr(model, 'class_weight'):
                    if model.class_weight is not None:
                        model.set_params(class_weight='balanced')
                
                # Train model
                model_clone = model.__class__(**model.get_params())
                model_clone.fit(X_train, y_train_target)
                target_models[target_name] = model_clone
                
                # Make predictions
                y_pred_train = model_clone.predict(X_train)
                y_pred_proba_train = model_clone.predict_proba(X_train)[:, 1] if hasattr(model_clone, 'predict_proba') else None
                
                # Calculate training metrics
                train_metrics = self._calculate_metrics(y_train_target, y_pred_train, y_pred_proba_train)
                
                # Cross-validation if requested
                cv_mean_auc = None
                cv_std_auc = None
                
                if use_cv and cv_folds > 1:
                    try:
                        cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
                        cv_scores = cross_val_score(model_clone, X_train, y_train_target, 
                                                  cv=cv, scoring='roc_auc', n_jobs=-1)
                        cv_mean_auc = cv_scores.mean()
                        cv_std_auc = cv_scores.std()
                        cv_results_all[target_name] = cv_scores
                        
                        logger.info(f"    Train AUC: {train_metrics.get('roc_auc', 0):.4f}, CV AUC: {cv_mean_auc:.4f} (±{cv_std_auc:.4f})")
                    except Exception as e:
                        logger.warning(f"    CV failed: {e}")
                        cv_mean_auc = 0.5
                        cv_std_auc = 0.0
                else:
                    logger.info(f"    Train AUC: {train_metrics.get('roc_auc', 0):.4f}")
                
                target_metrics[target_name] = {
                    'train_metrics': train_metrics,
                    'cv_mean_auc': cv_mean_auc,
                    'cv_std_auc': cv_std_auc,
                    'model': model_clone
                }
            
            # Calculate average metrics
            avg_train_auc = np.mean([m['train_metrics'].get('roc_auc', 0) for m in target_metrics.values()])
            avg_cv_auc = np.mean([m['cv_mean_auc'] for m in target_metrics.values()]) if use_cv else None
            
            # Store results
            self.results['model_comparison'][model_name] = {
                'avg_train_auc': avg_train_auc,
                'avg_cv_auc': avg_cv_auc,
                'models': target_models,
                'cv_results': cv_results_all if use_cv else {}
            }
            
            self.results['cv_scores'][model_name] = cv_results_all
            
            for target_name in target_names:
                self.results['target_results'][target_name][model_name] = target_metrics[target_name]
        
        return self.results
    
    def _calculate_metrics(self, y_true, y_pred, y_pred_proba=None):
        """Calculate comprehensive classification metrics."""
        metrics = {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'support': len(y_true)
        }
        
        # Calculate additional metrics if probabilities are available
        if y_pred_proba is not None:
            try:
                metrics['roc_auc'] = roc_auc_score(y_true, y_pred_proba)
                metrics['average_precision'] = average_precision_score(y_true, y_pred_proba)
            except Exception as e:
                logger.debug(f"Could not calculate probability metrics: {e}")
                metrics['roc_auc'] = 0.5
                metrics['average_precision'] = y_true.mean()
        
        # Confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        metrics['confusion_matrix'] = {'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp}
        metrics['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0
        
        return metrics
    
    def evaluate_on_test(self):
        """Evaluate all models on test set with comprehensive analysis."""
        logger.info("\n" + "=" * 70)
        logger.info("COMPREHENSIVE TEST SET EVALUATION")
        logger.info("=" * 70)
        
        X_test = self.modeling_data['X_test']
        y_test = self.modeling_data['y_test']
        target_names = self.modeling_data['target_names']
        
        test_results = {}
        detailed_predictions = {}
        
        for model_name in self.models.keys():
            logger.info(f"\nEvaluating {model_name} on test set...")
            
            model_info = self.results['model_comparison'][model_name]
            target_models = model_info['models']
            
            model_test_metrics = {}
            model_predictions = {}
            
            for target_idx, target_name in enumerate(target_names):
                model = target_models[target_name]
                y_test_target = y_test[:, target_idx]
                
                # Make predictions
                y_pred_test = model.predict(X_test)
                y_pred_proba_test = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
                
                # Calculate comprehensive metrics
                test_metrics = self._calculate_metrics(y_test_target, y_pred_test, y_pred_proba_test)
                
                # Store predictions for detailed analysis
                model_predictions[target_name] = {
                    'y_true': y_test_target,
                    'y_pred': y_pred_test,
                    'y_pred_proba': y_pred_proba_test
                }
                
                model_test_metrics[target_name] = test_metrics
                
                # Log key metrics
                logger.info(f"  {target_name}:")
                logger.info(f"    AUC: {test_metrics.get('roc_auc', 0):.4f}, F1: {test_metrics['f1']:.4f}")
                logger.info(f"    Precision: {test_metrics['precision']:.4f}, Recall: {test_metrics['recall']:.4f}")
                logger.info(f"    Specificity: {test_metrics.get('specificity', 0):.4f}")
            
            test_results[model_name] = model_test_metrics
            detailed_predictions[model_name] = model_predictions
        
        self.results['test_results'] = test_results
        self.results['detailed_predictions'] = detailed_predictions
        
        return test_results
    
    def analyze_feature_importance(self, top_n: int = 20):
        """Analyze feature importance across all tree-based models."""
        logger.info(f"\nAnalyzing feature importance (top {top_n} features)...")
        
        os.makedirs('reports/individual_full/feature_importance', exist_ok=True)
        
        feature_names = self.modeling_data['feature_names']
        self.feature_importance = {}
        
        # Models that support feature importance
        importance_models = ['Random_Forest', 'XGBoost', 'LightGBM', 
                            'Gradient_Boosting', 'Decision_Tree', 'CatBoost']
        
        for model_name in importance_models:
            if model_name in self.results['model_comparison']:
                try:
                    # Get model for first target
                    first_target = list(self.results['model_comparison'][model_name]['models'].keys())[0]
                    model = self.results['model_comparison'][model_name]['models'][first_target]
                    
                    if hasattr(model, 'feature_importances_'):
                        importances = model.feature_importances_
                        
                        # Create DataFrame
                        importance_df = pd.DataFrame({
                            'Feature': feature_names,
                            'Importance': importances
                        }).sort_values('Importance', ascending=False).head(top_n)
                        
                        self.feature_importance[model_name] = importance_df
                        
                        # Save to CSV
                        importance_df.to_csv(f'reports/individual_full/feature_importance/{model_name}_importance.csv', index=False)
                        
                        # Create visualization
                        self._plot_feature_importance(importance_df, model_name, top_n)
                        
                        logger.info(f"  {model_name}: Top 3 features - {', '.join(importance_df['Feature'].head(3).tolist())}")
                    
                except Exception as e:
                    logger.warning(f"  Could not analyze feature importance for {model_name}: {e}")
        
        return self.feature_importance
    
    def _plot_feature_importance(self, importance_df, model_name, top_n):
        """Create feature importance visualization."""
        plt.figure(figsize=(12, 8))
        plt.barh(range(len(importance_df)), importance_df['Importance'])
        plt.yticks(range(len(importance_df)), importance_df['Feature'], fontsize=9)
        plt.xlabel('Feature Importance')
        plt.title(f'Top {top_n} Features - {model_name}')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.savefig(f'reports/individual_full/feature_importance/{model_name}_importance.png', 
                   dpi=150, bbox_inches='tight')
        plt.close()
    
    def create_comparison_report(self):
        """Create comprehensive model comparison report."""
        logger.info("\nCreating comprehensive model comparison report...")
        
        os.makedirs('reports/individual_full/model_comparison', exist_ok=True)
        
        # Create comparison DataFrame
        comparison_data = []
        
        for model_name, model_info in self.results['model_comparison'].items():
            row = {
                'Model': model_name,
                'Avg_Train_AUC': model_info['avg_train_auc'],
                'Avg_CV_AUC': model_info.get('avg_cv_auc', 0)
            }
            
            # Add test metrics if available
            if 'test_results' in self.results and model_name in self.results['test_results']:
                test_aucs = [m.get('roc_auc', 0) for m in self.results['test_results'][model_name].values()]
                test_f1s = [m.get('f1', 0) for m in self.results['test_results'][model_name].values()]
                row['Avg_Test_AUC'] = np.mean(test_aucs)
                row['Avg_Test_F1'] = np.mean(test_f1s)
            
            comparison_data.append(row)
        
        comparison_df = pd.DataFrame(comparison_data)
        
        # Sort by CV AUC if available, otherwise by Test AUC
        if 'Avg_CV_AUC' in comparison_df.columns:
            comparison_df = comparison_df.sort_values('Avg_CV_AUC', ascending=False)
        elif 'Avg_Test_AUC' in comparison_df.columns:
            comparison_df = comparison_df.sort_values('Avg_Test_AUC', ascending=False)
        
        # Save comparison
        comparison_path = 'reports/individual_full/model_comparison/model_comparison.csv'
        comparison_df.to_csv(comparison_path, index=False)
        
        # Create visual comparison
        self._plot_model_comparison(comparison_df)
        
        logger.info("\nModel Comparison (sorted by performance):")
        print(comparison_df.to_string(index=False))
        
        return comparison_df
    
    def _plot_model_comparison(self, comparison_df):
        """Create model comparison visualization."""
        if 'Avg_CV_AUC' in comparison_df.columns:
            metrics = ['Avg_Train_AUC', 'Avg_CV_AUC']
            if 'Avg_Test_AUC' in comparison_df.columns:
                metrics.append('Avg_Test_AUC')
            
            fig, axes = plt.subplots(1, 2, figsize=(15, 6))
            
            # Bar chart
            x = range(len(comparison_df))
            width = 0.25
            
            for i, metric in enumerate(metrics):
                axes[0].bar([pos + i*width for pos in x], 
                           comparison_df[metric], 
                           width=width, 
                           label=metric.replace('_', ' '))
            
            axes[0].set_xlabel('Models')
            axes[0].set_ylabel('AUC Score')
            axes[0].set_title('Model Performance Comparison')
            axes[0].set_xticks([pos + width for pos in x])
            axes[0].set_xticklabels(comparison_df['Model'], rotation=45, ha='right')
            axes[0].legend()
            axes[0].grid(True, alpha=0.3)
            
            # Radar chart for top 5 models
            top_models = comparison_df.head(5)
            metrics_radar = ['Avg_Train_AUC', 'Avg_CV_AUC', 'Avg_Test_AUC'] \
                           if 'Avg_Test_AUC' in top_models.columns else ['Avg_Train_AUC', 'Avg_CV_AUC']
            
            angles = np.linspace(0, 2*np.pi, len(metrics_radar), endpoint=False).tolist()
            angles += angles[:1]  # Close the polygon
            
            for idx, row in top_models.iterrows():
                values = [row[metric] for metric in metrics_radar]
                values += values[:1]  # Close the polygon
                axes[1].plot(angles, values, 'o-', label=row['Model'])
                axes[1].fill(angles, values, alpha=0.25)
            
            axes[1].set_xticks(angles[:-1])
            axes[1].set_xticklabels([m.replace('_', '\n') for m in metrics_radar])
            axes[1].set_ylim(0, 1)
            axes[1].set_title('Top 5 Models - Radar Chart')
            axes[1].legend(loc='upper right', bbox_to_anchor=(1.3, 1))
            axes[1].grid(True)
            
            plt.tight_layout()
            plt.savefig('reports/individual_full/model_comparison/model_comparison_visualization.png', 
                       dpi=150, bbox_inches='tight')
            plt.close()
    
    def identify_best_models(self):
        """Identify best model for each target and overall."""
        logger.info("\nIdentifying best models...")
        
        self.best_models = {
            'overall': {},
            'by_target': {},
            'by_metric': {}
        }
        
        # Get available metrics
        available_metrics = []
        if self.results.get('test_results'):
            first_model = list(self.results['test_results'].keys())[0]
            first_target = list(self.results['test_results'][first_model].keys())[0]
            available_metrics = list(self.results['test_results'][first_model][first_target].keys())
        
        # Find best overall model (by average test AUC)
        if 'test_results' in self.results:
            model_avg_auc = {}
            for model_name, target_results in self.results['test_results'].items():
                auc_scores = [metrics.get('roc_auc', 0) for metrics in target_results.values()]
                model_avg_auc[model_name] = np.mean(auc_scores)
            
            if model_avg_auc:
                best_overall = max(model_avg_auc.items(), key=lambda x: x[1])
                self.best_models['overall'] = {
                    'model': best_overall[0],
                    'auc': best_overall[1],
                    'metric': 'Average Test AUC'
                }
                logger.info(f"Best overall model: {best_overall[0]} (AUC: {best_overall[1]:.4f})")
        
        # Find best model for each target
        if 'test_results' in self.results:
            for target_name in self.results['test_results'][list(self.results['test_results'].keys())[0]]:
                best_model = None
                best_auc = 0
                
                for model_name, target_results in self.results['test_results'].items():
                    if target_name in target_results:
                        auc = target_results[target_name].get('roc_auc', 0)
                        if auc > best_auc:
                            best_auc = auc
                            best_model = model_name
                
                if best_model:
                    self.best_models['by_target'][target_name] = {
                        'model': best_model,
                        'auc': best_auc
                    }
                    logger.info(f"Best model for {target_name}: {best_model} (AUC: {best_auc:.4f})")
        
        return self.best_models
    
    def save_models_and_results(self, output_dir: str = 'models/individual_full'):
        """Save trained models and all results."""
        logger.info(f"\nSaving models and results to {output_dir}...")
        
        os.makedirs(output_dir, exist_ok=True)
        
        # Save each model
        for model_name, model_info in self.results['model_comparison'].items():
            model_dir = os.path.join(output_dir, model_name)
            os.makedirs(model_dir, exist_ok=True)
            
            for target_name, model in model_info['models'].items():
                model_path = os.path.join(model_dir, f"{target_name}.pkl")
                joblib.dump(model, model_path)
            
            logger.info(f"  Saved {model_name}")
        
        # Save preprocessor if available
        if 'preprocessor' in self.modeling_data:
            preprocessor_path = os.path.join(output_dir, 'preprocessor.pkl')
            joblib.dump(self.modeling_data['preprocessor'], preprocessor_path)
            logger.info(f"  Saved preprocessor")
        
        # Save results
        results_path = os.path.join(output_dir, 'training_results.pkl')
        
        # Convert to JSON-serializable format
        results_serializable = {}
        for key, value in self.results.items():
            if isinstance(value, dict):
                results_serializable[key] = {}
                for subkey, subvalue in value.items():
                    if isinstance(subvalue, np.ndarray):
                        results_serializable[key][subkey] = subvalue.tolist()
                    elif hasattr(subvalue, '__dict__'):  # Skip objects
                        continue
                    else:
                        results_serializable[key][subkey] = subvalue
            else:
                results_serializable[key] = value
        
        joblib.dump(self.results, results_path)
        
        # Also save as JSON for readability
        results_json_path = os.path.join(output_dir, 'training_results.json')
        with open(results_json_path, 'w') as f:
            json.dump(results_serializable, f, indent=2, default=str)
        
        # Save best models
        best_models_path = os.path.join(output_dir, 'best_models.json')
        with open(best_models_path, 'w') as f:
            json.dump(self.best_models, f, indent=2)
        
        logger.info(f"All models and results saved to {output_dir}/")
    
    def generate_final_report(self):
        """Generate comprehensive final report."""
        logger.info("\n" + "=" * 70)
        logger.info("GENERATING FINAL COMPREHENSIVE REPORT")
        logger.info("=" * 70)
        
        report_dir = 'reports/individual_full/final_report'
        os.makedirs(report_dir, exist_ok=True)
        
        report_content = """
        ============================================
        FINAL MODELING REPORT - INDIVIDUAL-LEVEL ANALYSIS
        ============================================
        
        Project: Multi-Disease Prediction using BRFSS and EPA AQI Data
        Analysis Level: Individual (400K+ samples)
        Date: {date}
        
        """.format(date=pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'))
        
        # Dataset Summary
        report_content += "\nDATASET SUMMARY\n"
        report_content += "=" * 50 + "\n"
        report_content += f"Total samples: {self.modeling_data['X_train'].shape[0] + self.modeling_data['X_test'].shape[0]:,}\n"
        report_content += f"Training samples: {self.modeling_data['X_train'].shape[0]:,}\n"
        report_content += f"Testing samples: {self.modeling_data['X_test'].shape[0]:,}\n"
        report_content += f"Number of features: {len(self.modeling_data['feature_names'])}\n"
        report_content += f"Target diseases: {', '.join(self.modeling_data['target_names'])}\n"
        
        # Disease prevalence
        report_content += "\nDisease Prevalence in Training Set:\n"
        for i, target in enumerate(self.modeling_data['target_names']):
            prevalence = self.modeling_data['y_train'][:, i].mean() * 100
            report_content += f"  {target}: {prevalence:.2f}%\n"
        
        # Model Performance Summary
        report_content += "\n\nMODEL PERFORMANCE SUMMARY\n"
        report_content += "=" * 50 + "\n"
        
        if self.best_models.get('overall'):
            best = self.best_models['overall']
            report_content += f"\nBest Overall Model: {best['model']}\n"
            report_content += f"Performance ({best['metric']}): {best['auc']:.4f}\n"
        
        if self.best_models.get('by_target'):
            report_content += "\nBest Models by Disease:\n"
            for disease, info in self.best_models['by_target'].items():
                report_content += f"  {disease}: {info['model']} (AUC: {info['auc']:.4f})\n"
        
        # Key Findings
        report_content += "\n\nKEY FINDINGS\n"
        report_content += "=" * 50 + "\n"
        
        # Top features from importance analysis
        if self.feature_importance:
            report_content += "\nTop Predictive Features (from Random Forest):\n"
            if 'Random_Forest' in self.feature_importance:
                top_features = self.feature_importance['Random_Forest'].head(10)
                for idx, row in top_features.iterrows():
                    report_content += f"  {row['Feature']}: {row['Importance']:.4f}\n"
        
        # Performance Insights
        report_content += "\nPerformance Insights:\n"
        if 'test_results' in self.results:
            # Calculate average performance across all models
            all_aucs = []
            for model_results in self.results['test_results'].values():
                for target_results in model_results.values():
                    all_aucs.append(target_results.get('roc_auc', 0))
            
            if all_aucs:
                report_content += f"  Average AUC across all models: {np.mean(all_aucs):.4f}\n"
                report_content += f"  Standard deviation: {np.std(all_aucs):.4f}\n"
                report_content += f"  Range: [{min(all_aucs):.4f}, {max(all_aucs):.4f}]\n"
        
        # Recommendations
        report_content += "\n\nRECOMMENDATIONS FOR PRODUCTION\n"
        report_content += "=" * 50 + "\n"
        
        if self.best_models.get('overall'):
            best_model = self.best_models['overall']['model']
            
            report_content += f"\n1. Primary Model: {best_model}\n"
            report_content += "   - Use for all production predictions\n"
            report_content += "   - Consider model interpretability vs accuracy trade-off\n\n"
            
            report_content += "2. Model Deployment Strategy:\n"
            report_content += "   - Deploy as microservice with REST API\n"
            report_content += "   - Implement A/B testing for model updates\n"
            report_content += "   - Monitor model drift with regular validation\n\n"
            
            report_content += "3. Future Improvements:\n"
            report_content += "   - Add temporal data for trend analysis\n"
            report_content += "   - Incorporate more environmental factors\n"
            report_content += "   - Implement ensemble of top 3 models\n"
            report_content += "   - Add explainable AI (SHAP values)\n"
        
        # Save report
        report_path = os.path.join(report_dir, 'final_report.txt')
        with open(report_path, 'w') as f:
            f.write(report_content)
        
        logger.info(f"Final report saved to {report_path}")
        
        return report_content


def main():
    """Main execution function."""
    logger.info("=" * 70)
    logger.info("ENHANCED INDIVIDUAL-LEVEL MODEL TRAINING PIPELINE")
    logger.info("=" * 70)
    
    # Create output directories
    os.makedirs('models/individual_full', exist_ok=True)
    os.makedirs('reports/individual_full', exist_ok=True)
    os.makedirs('reports/individual_full/feature_importance', exist_ok=True)
    os.makedirs('reports/individual_full/model_comparison', exist_ok=True)
    
    try:
        # Initialize enhanced trainer
        trainer = IndividualModelTrainerEnhanced()
        
        # Step 1: Load data
        logger.info("\n[STEP 1] Loading enhanced individual data...")
        if not trainer.load_data():
            return
        
        # Step 2: Create models
        logger.info("\n[STEP 2] Creating advanced models...")
        trainer.create_models(use_advanced=True)
        
        # Step 3: Train models with cross-validation
        logger.info("\n[STEP 3] Training models with cross-validation...")
        trainer.train_models(cv_folds=3, use_cv=True)
        
        # Step 4: Evaluate on test set
        logger.info("\n[STEP 4] Evaluating models on test set...")
        trainer.evaluate_on_test()
        
        # Step 5: Identify best models
        logger.info("\n[STEP 5] Identifying best models...")
        trainer.identify_best_models()
        
        # Step 6: Analyze feature importance
        logger.info("\n[STEP 6] Analyzing feature importance...")
        trainer.analyze_feature_importance(top_n=25)
        
        # Step 7: Create comparison report
        logger.info("\n[STEP 7] Creating model comparison report...")
        comparison_df = trainer.create_comparison_report()
        
        # Step 8: Save everything
        logger.info("\n[STEP 8] Saving models and results...")
        trainer.save_models_and_results()
        
        # Step 9: Generate final report
        logger.info("\n[STEP 9] Generating final report...")
        trainer.generate_final_report()
        
        # Final summary
        logger.info("\n" + "=" * 70)
        logger.info("TRAINING PIPELINE COMPLETE")
        logger.info("=" * 70)
        
        if trainer.best_models.get('overall'):
            best = trainer.best_models['overall']
            logger.info(f"🎯 BEST OVERALL MODEL: {best['model']}")
            logger.info(f"   Average Test AUC: {best['auc']:.4f}")
        
        logger.info(f"\n📊 Dataset Statistics:")
        logger.info(f"   Training samples: {trainer.modeling_data['X_train'].shape[0]:,}")
        logger.info(f"   Testing samples: {trainer.modeling_data['X_test'].shape[0]:,}")
        logger.info(f"   Features: {len(trainer.modeling_data['feature_names'])}")
        
        logger.info(f"\n📁 Outputs Saved:")
        logger.info(f"   Models: models/individual_full/")
        logger.info(f"   Reports: reports/individual_full/")
        logger.info(f"   Feature importance analysis")
        logger.info(f"   Model comparison visualizations")
        logger.info(f"   Comprehensive final report")
        
        return trainer
        
    except Exception as e:
        logger.error(f"Training pipeline failed: {e}")
        import traceback
        traceback.print_exc()
        return None


if __name__ == "__main__":
    trainer = main()

2025-12-14 22:36:44,722 - INFO - ======================================================================
2025-12-14 22:36:44,723 - INFO - ENHANCED INDIVIDUAL-LEVEL MODEL TRAINING PIPELINE
2025-12-14 22:36:44,724 - INFO - ======================================================================
2025-12-14 22:36:44,725 - INFO - 
[STEP 1] Loading enhanced individual data...
2025-12-14 22:36:44,725 - INFO - Loading modeling data from data/processed/individual_full/modeling_data.pkl
2025-12-14 22:36:45,848 - INFO - ======================================================================
2025-12-14 22:36:45,848 - INFO - DATASET OVERVIEW
2025-12-14 22:36:45,849 - INFO - ======================================================================
2025-12-14 22:36:45,849 - INFO - X_train shape: (346658, 65)
2025-12-14 22:36:45,849 - INFO - X_test shape: (86665, 65)
2025-12-14 22:36:45,850 - INFO - Features: 65
2025-12-14 22:36:45,850 - INFO - Targets: Diabetes, HeartDisease, Stroke, Asthma
2025-12-14 22:36